# Imports

In [50]:
import logging, warnings; logging.getLogger().setLevel(logging.ERROR);
warnings.filterwarnings("ignore")

import scanpy as sc
import scanpy.external as sce
import numpy as np
import pandas as pd
import re
from pathlib import Path 
import os

import warnings, scipy.sparse as sp, matplotlib, matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.pyplot import rc_context
import matplotlib.font_manager
import matplotlib.lines as lines


pd.set_option('display.max_rows', 200)

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = 'Arial'
matplotlib.rc('font', size=12)

sc.settings.n_jobs=-1
sc.set_figure_params(dpi=80, dpi_save=300, color_map='Spectral_r', vector_friendly=True, transparent=True)
sc.settings.figdir = '../../../1_outputs/0_figures'
sc.settings.verbosity = 1 # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()

%matplotlib inline 
%config InlineBackend.figure_format = 'retina'

In [2]:
# preset color palettes and color maps
user_defined_palette =  [ '#F6222E', '#16FF32', '#3283FE', '#FEAF16', '#BDCDFF', '#3B00FB', '#1CFFCE', '#C075A6', '#F8A19F', '#B5EFB5', '#FBE426', '#C4451C', 
                          '#2ED9FF', '#c1c119', '#8b0000', '#FE00FA', '#1CBE4F', '#1C8356', '#0e452b', '#AA0DFE', '#B5EFB5', '#325A9B', '#90AD1C']

user_defined_cmap_markers = LinearSegmentedColormap.from_list('mycmap', ["#E6E6FF", "#CCCCFF", "#B2B2FF", "#9999FF",  "#6666FF",   "#3333FF", "#0000FF"])
user_defined_cmap_degs = LinearSegmentedColormap.from_list('mycmap', ["#0000FF", "#3333FF", "#6666FF", "#9999FF", "#B2B2FF", "#CCCCFF", "#E6E6FF", "#E6FFE6", "#CCFFCC", "#B2FFB2", "#99FF99", "#66FF66", "#33FF33", "#00FF00"])

In [3]:
def sanitize_sheet_name(name):
    return name.replace(':', '_')

In [4]:
pwd

'/Users/mkaur/github/2_tff1/1_pyzone/0_notebooks/0_scRNA/2_deg'

In [5]:
deg_outputs = '../../../1_outputs/1_degs/' 
h5ad = '../../../4_h5ad/'

In [6]:
## read in h5ad 

adata = sc.read_h5ad(h5ad + "8_tregs.h5ad")
adata


AnnData object with n_obs × n_vars = 10273 × 23867
    obs: 'condition', 'tissue', 'sample', 'cell_type', 'tissues_ordered', 'spectra50', 'treg_regen', 'treg_supress', 'treg_naive', 'treg_celltypes'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'cell_type_colors', 'condition_colors', 'log1p', 'neighbors', 'pca', 'sample_colors', 'tissue_colors', 'tissues_ordered_colors', 'treg_celltypes_colors', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'raw_data'
    obsp: 'connectivities', 'distances'

# Thymus Tregs vs Rest

In [8]:
adata

AnnData object with n_obs × n_vars = 10273 × 23867
    obs: 'condition', 'tissue', 'sample', 'cell_type', 'tissues_ordered', 'spectra50', 'treg_regen', 'treg_supress', 'treg_naive', 'treg_celltypes'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'cell_type_colors', 'condition_colors', 'log1p', 'neighbors', 'pca', 'sample_colors', 'tissue_colors', 'tissues_ordered_colors', 'treg_celltypes_colors', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'raw_data'
    obsp: 'connectivities', 'distances'

In [14]:
adata.obs['treg_celltypes']

TGCGGGTGTGAGTGAC-1@gvhd_thymus       1:Quiescent
CGTTCTGAGATCGGGT-1@gvhd_spleen    0:Regenerative
TGCGGGTCATGGTTGT-1@gvhd_skin         1:Quiescent
AGTGAGGAGATCTGAA-1@gvhd_spleen       1:Quiescent
ACTATCTCACAGGAGT-1@ss_spleen         1:Quiescent
                                       ...      
TTGGCAAAGGCCCTTG-1@ss_liver       0:Regenerative
AAGGAGCGTCTAGGTT-1@gvhd_spleen    0:Regenerative
CCGTGGAGTCCTCCAT-1@ss_liver          1:Quiescent
GCAATCAGTTAAAGAC-1@ss_bm             1:Quiescent
ACTGAGTGTTAGATGA-1@gvhd_thymus       1:Quiescent
Name: treg_celltypes, Length: 10273, dtype: category
Categories (2, object): ['0:Regenerative', '1:Quiescent']

In [11]:
regen = adata[adata.obs['treg_celltypes'].isin(['0:Regenerative'])].copy()
regen.obs['treg_celltypes']

CGTTCTGAGATCGGGT-1@gvhd_spleen    0:Regenerative
CCGTTCACAGTCAGCC-1@gvhd_spleen    0:Regenerative
TCCCGATAGTGCGATG-1@ss_skin        0:Regenerative
CGTGTCTTCCACGCAG-1@gvhd_spleen    0:Regenerative
TGACGGCGTTGCGTTA-1@ss_skin        0:Regenerative
                                       ...      
ACTGTCCAGCTAGTCT-1@ss_bm          0:Regenerative
TTGCGTCCACGACGAA-1@hct_spleen     0:Regenerative
GATCGCGAGCGAGAAA-1@hct_bm         0:Regenerative
TTGGCAAAGGCCCTTG-1@ss_liver       0:Regenerative
AAGGAGCGTCTAGGTT-1@gvhd_spleen    0:Regenerative
Name: treg_celltypes, Length: 5354, dtype: category
Categories (1, object): ['0:Regenerative']

In [12]:
writer = pd.ExcelWriter(deg_outputs + '0_thymus_treg_vs_rest.xlsx', engine='xlsxwriter')

In [16]:
sc.tl.rank_genes_groups(regen, 'tissues_ordered', groups=['0:Thymus'], reference='rest', method='t-test', use_raw=False)
result = regen.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names', 'scores', 'logfoldchanges', 'pvals_adj']})

logfoldchange_cols = [col for col in df.columns if col.endswith('_l')]  # Logfoldchange columns (third column)
pval_cols = [col for col in df.columns if col.endswith('_p')]  # Adjusted p-value columns (fourth column)

if logfoldchange_cols and pval_cols:
    # Assuming single comparison, use the first detected columns
    df['Is Significant'] = (df[logfoldchange_cols[0]].abs() > 0.585) & (df[pval_cols[0]] < 0.05)

df.to_excel(writer, sheet_name=sanitize_sheet_name(groups[0]))
        
writer.close()

## Remove SS Condition

In [7]:
regen = adata[adata.obs['treg_celltypes'].isin(['0:Regenerative'])].copy()
regen.obs['treg_celltypes']

CGTTCTGAGATCGGGT-1@gvhd_spleen    0:Regenerative
CCGTTCACAGTCAGCC-1@gvhd_spleen    0:Regenerative
TCCCGATAGTGCGATG-1@ss_skin        0:Regenerative
CGTGTCTTCCACGCAG-1@gvhd_spleen    0:Regenerative
TGACGGCGTTGCGTTA-1@ss_skin        0:Regenerative
                                       ...      
ACTGTCCAGCTAGTCT-1@ss_bm          0:Regenerative
TTGCGTCCACGACGAA-1@hct_spleen     0:Regenerative
GATCGCGAGCGAGAAA-1@hct_bm         0:Regenerative
TTGGCAAAGGCCCTTG-1@ss_liver       0:Regenerative
AAGGAGCGTCTAGGTT-1@gvhd_spleen    0:Regenerative
Name: treg_celltypes, Length: 5354, dtype: category
Categories (1, object): ['0:Regenerative']

In [15]:
regen.obs['condition']

CGTTCTGAGATCGGGT-1@gvhd_spleen    gvhd
CCGTTCACAGTCAGCC-1@gvhd_spleen    gvhd
TCCCGATAGTGCGATG-1@ss_skin          ss
CGTGTCTTCCACGCAG-1@gvhd_spleen    gvhd
TGACGGCGTTGCGTTA-1@ss_skin          ss
                                  ... 
ACTGTCCAGCTAGTCT-1@ss_bm            ss
TTGCGTCCACGACGAA-1@hct_spleen      hct
GATCGCGAGCGAGAAA-1@hct_bm          hct
TTGGCAAAGGCCCTTG-1@ss_liver         ss
AAGGAGCGTCTAGGTT-1@gvhd_spleen    gvhd
Name: condition, Length: 5354, dtype: category
Categories (3, object): ['gvhd', 'hct', 'ss']

In [10]:
regen_nonss = regen[~regen.obs['condition'].isin(['ss'])].copy()
regen_nonss 

AnnData object with n_obs × n_vars = 3082 × 23867
    obs: 'condition', 'tissue', 'sample', 'cell_type', 'tissues_ordered', 'spectra50', 'treg_regen', 'treg_supress', 'treg_naive', 'treg_celltypes'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'cell_type_colors', 'condition_colors', 'log1p', 'neighbors', 'pca', 'sample_colors', 'tissue_colors', 'tissues_ordered_colors', 'treg_celltypes_colors', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'raw_data'
    obsp: 'connectivities', 'distances'

In [51]:
writer = pd.ExcelWriter(deg_outputs + '1_nonss_thymus_treg_vs_rest.xlsx', engine='xlsxwriter')

In [53]:
sc.tl.rank_genes_groups(regen_nonss, 'tissues_ordered', groups=['0:Thymus'], reference='rest', method='t-test', use_raw=False)
result = regen_nonss.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names', 'scores', 'logfoldchanges', 'pvals_adj']})

logfoldchange_cols = [col for col in df.columns if col.endswith('_l')]  # Logfoldchange columns (third column)
pval_cols = [col for col in df.columns if col.endswith('_p')]  # Adjusted p-value columns (fourth column)

if logfoldchange_cols and pval_cols:
    # Assuming single comparison, use the first detected columns
    df['Is Significant'] = (df[logfoldchange_cols[0]].abs() > 0.585) & (df[pval_cols[0]] < 10e-6)

df.to_excel(writer, sheet_name=sanitize_sheet_name(groups[0]))
        
writer.close()

## HCT Only

In [16]:
adata

AnnData object with n_obs × n_vars = 10273 × 23867
    obs: 'condition', 'tissue', 'sample', 'cell_type', 'tissues_ordered', 'spectra50', 'treg_regen', 'treg_supress', 'treg_naive', 'treg_celltypes'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'cell_type_colors', 'condition_colors', 'log1p', 'neighbors', 'pca', 'sample_colors', 'tissue_colors', 'tissues_ordered_colors', 'treg_celltypes_colors', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'raw_data'
    obsp: 'connectivities', 'distances'

In [17]:
regen = adata[adata.obs['treg_celltypes'].isin(['0:Regenerative'])].copy()
regen.obs['treg_celltypes']

CGTTCTGAGATCGGGT-1@gvhd_spleen    0:Regenerative
CCGTTCACAGTCAGCC-1@gvhd_spleen    0:Regenerative
TCCCGATAGTGCGATG-1@ss_skin        0:Regenerative
CGTGTCTTCCACGCAG-1@gvhd_spleen    0:Regenerative
TGACGGCGTTGCGTTA-1@ss_skin        0:Regenerative
                                       ...      
ACTGTCCAGCTAGTCT-1@ss_bm          0:Regenerative
TTGCGTCCACGACGAA-1@hct_spleen     0:Regenerative
GATCGCGAGCGAGAAA-1@hct_bm         0:Regenerative
TTGGCAAAGGCCCTTG-1@ss_liver       0:Regenerative
AAGGAGCGTCTAGGTT-1@gvhd_spleen    0:Regenerative
Name: treg_celltypes, Length: 5354, dtype: category
Categories (1, object): ['0:Regenerative']

In [ ]:
regen_hct = regen[regen.obs['condition'].isin(['hct'])].copy()
regen_hct.obs['condition']

AGATTGCAGTGTACCT-1@hct_thymus    hct
AAGGTTCAGCTTCGCG-1@hct_spleen    hct
AGAATAGCAAGAAAGG-1@hct_bm        hct
GAGGTGAGTCAGATAA-1@hct_bm        hct
GGGCACTCAGACGCAA-1@hct_thymus    hct
                                ... 
GGGTCTGCAAGAGGCT-1@hct_spleen    hct
CGGACACTCTCCAACC-1@hct_thymus    hct
GGATGTTTCCTTGCCA-1@hct_bm        hct
TTGCGTCCACGACGAA-1@hct_spleen    hct
GATCGCGAGCGAGAAA-1@hct_bm        hct
Name: condition, Length: 1515, dtype: category
Categories (1, object): ['hct']

In [26]:
writer = pd.ExcelWriter(deg_outputs + '2_hct_thymus_treg_vs_rest.xlsx', engine='xlsxwriter')

In [27]:
sc.tl.rank_genes_groups(regen_hct, 'tissues_ordered', groups=['0:Thymus'], reference='rest', method='t-test', use_raw=False)
result = regen_hct.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names', 'scores', 'logfoldchanges', 'pvals_adj']})

logfoldchange_cols = [col for col in df.columns if col.endswith('_l')]  # Logfoldchange columns (third column)
pval_cols = [col for col in df.columns if col.endswith('_p')]  # Adjusted p-value columns (fourth column)

if logfoldchange_cols and pval_cols:
    # Assuming single comparison, use the first detected columns
    df['Is Significant'] = (df[logfoldchange_cols[0]].abs() > 0.585) & (df[pval_cols[0]] < 10e-6)

df.to_excel(writer, sheet_name=sanitize_sheet_name(groups[0]))
        
writer.close()

## GVHD only

In [29]:
adata

AnnData object with n_obs × n_vars = 10273 × 23867
    obs: 'condition', 'tissue', 'sample', 'cell_type', 'tissues_ordered', 'spectra50', 'treg_regen', 'treg_supress', 'treg_naive', 'treg_celltypes'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'cell_type_colors', 'condition_colors', 'log1p', 'neighbors', 'pca', 'sample_colors', 'tissue_colors', 'tissues_ordered_colors', 'treg_celltypes_colors', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'raw_data'
    obsp: 'connectivities', 'distances'

In [30]:
regen = adata[adata.obs['treg_celltypes'].isin(['0:Regenerative'])].copy()
regen.obs['treg_celltypes']

CGTTCTGAGATCGGGT-1@gvhd_spleen    0:Regenerative
CCGTTCACAGTCAGCC-1@gvhd_spleen    0:Regenerative
TCCCGATAGTGCGATG-1@ss_skin        0:Regenerative
CGTGTCTTCCACGCAG-1@gvhd_spleen    0:Regenerative
TGACGGCGTTGCGTTA-1@ss_skin        0:Regenerative
                                       ...      
ACTGTCCAGCTAGTCT-1@ss_bm          0:Regenerative
TTGCGTCCACGACGAA-1@hct_spleen     0:Regenerative
GATCGCGAGCGAGAAA-1@hct_bm         0:Regenerative
TTGGCAAAGGCCCTTG-1@ss_liver       0:Regenerative
AAGGAGCGTCTAGGTT-1@gvhd_spleen    0:Regenerative
Name: treg_celltypes, Length: 5354, dtype: category
Categories (1, object): ['0:Regenerative']

In [31]:
regen_gvhd = regen[regen.obs['condition'].isin(['gvhd'])].copy()
regen_gvhd.obs['condition']

CGTTCTGAGATCGGGT-1@gvhd_spleen    gvhd
CCGTTCACAGTCAGCC-1@gvhd_spleen    gvhd
CGTGTCTTCCACGCAG-1@gvhd_spleen    gvhd
GTGGGTCTCCAGAGGA-1@gvhd_spleen    gvhd
GCTCTGTAGCGCCTTG-1@gvhd_thymus    gvhd
                                  ... 
AAGGTTCAGGCATGTG-1@gvhd_thymus    gvhd
CGTAGGCTCTTAGCCC-1@gvhd_thymus    gvhd
TACAGTGAGTTGAGAT-1@gvhd_thymus    gvhd
TGGACGCAGCGTTCCG-1@gvhd_thymus    gvhd
AAGGAGCGTCTAGGTT-1@gvhd_spleen    gvhd
Name: condition, Length: 1567, dtype: category
Categories (1, object): ['gvhd']

In [32]:
writer = pd.ExcelWriter(deg_outputs + '3_gvhd_thymus_treg_vs_rest.xlsx', engine='xlsxwriter')

In [33]:
sc.tl.rank_genes_groups(regen_gvhd, 'tissues_ordered', groups=['0:Thymus'], reference='rest', method='t-test', use_raw=False)
result = regen_gvhd.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names', 'scores', 'logfoldchanges', 'pvals_adj']})

logfoldchange_cols = [col for col in df.columns if col.endswith('_l')]  # Logfoldchange columns (third column)
pval_cols = [col for col in df.columns if col.endswith('_p')]  # Adjusted p-value columns (fourth column)

if logfoldchange_cols and pval_cols:
    # Assuming single comparison, use the first detected columns
    df['Is Significant'] = (df[logfoldchange_cols[0]].abs() > 0.585) & (df[pval_cols[0]] < 10e-6)

df.to_excel(writer, sheet_name=sanitize_sheet_name(groups[0]))
        
writer.close()

## Steady State Only

In [34]:
adata

AnnData object with n_obs × n_vars = 10273 × 23867
    obs: 'condition', 'tissue', 'sample', 'cell_type', 'tissues_ordered', 'spectra50', 'treg_regen', 'treg_supress', 'treg_naive', 'treg_celltypes'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'cell_type_colors', 'condition_colors', 'log1p', 'neighbors', 'pca', 'sample_colors', 'tissue_colors', 'tissues_ordered_colors', 'treg_celltypes_colors', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'raw_data'
    obsp: 'connectivities', 'distances'

In [35]:
regen = adata[adata.obs['treg_celltypes'].isin(['0:Regenerative'])].copy()
regen.obs['treg_celltypes']

CGTTCTGAGATCGGGT-1@gvhd_spleen    0:Regenerative
CCGTTCACAGTCAGCC-1@gvhd_spleen    0:Regenerative
TCCCGATAGTGCGATG-1@ss_skin        0:Regenerative
CGTGTCTTCCACGCAG-1@gvhd_spleen    0:Regenerative
TGACGGCGTTGCGTTA-1@ss_skin        0:Regenerative
                                       ...      
ACTGTCCAGCTAGTCT-1@ss_bm          0:Regenerative
TTGCGTCCACGACGAA-1@hct_spleen     0:Regenerative
GATCGCGAGCGAGAAA-1@hct_bm         0:Regenerative
TTGGCAAAGGCCCTTG-1@ss_liver       0:Regenerative
AAGGAGCGTCTAGGTT-1@gvhd_spleen    0:Regenerative
Name: treg_celltypes, Length: 5354, dtype: category
Categories (1, object): ['0:Regenerative']

In [36]:
regen_ss = regen[regen.obs['condition'].isin(['ss'])].copy()
regen_ss.obs['condition']

TCCCGATAGTGCGATG-1@ss_skin     ss
TGACGGCGTTGCGTTA-1@ss_skin     ss
CAGTAACCACAAGACG-1@ss_bm       ss
GTTAAGCCAGGATTGG-1@ss_bm       ss
TGACGGCTCGCCAAAT-1@ss_skin     ss
                               ..
TGACTAGAGTTAAGTG-1@ss_skin     ss
TTGGAACCATCGACGC-1@ss_skin     ss
TCACGAAGTTAAAGAC-1@ss_skin     ss
ACTGTCCAGCTAGTCT-1@ss_bm       ss
TTGGCAAAGGCCCTTG-1@ss_liver    ss
Name: condition, Length: 2272, dtype: category
Categories (1, object): ['ss']

In [37]:
writer = pd.ExcelWriter(deg_outputs + '4_ss_thymus_treg_vs_rest.xlsx', engine='xlsxwriter')

In [38]:
sc.tl.rank_genes_groups(regen_ss, 'tissues_ordered', groups=['0:Thymus'], reference='rest', method='t-test', use_raw=False)
result = regen_ss.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names', 'scores', 'logfoldchanges', 'pvals_adj']})

logfoldchange_cols = [col for col in df.columns if col.endswith('_l')]  # Logfoldchange columns (third column)
pval_cols = [col for col in df.columns if col.endswith('_p')]  # Adjusted p-value columns (fourth column)

if logfoldchange_cols and pval_cols:
    # Assuming single comparison, use the first detected columns
    df['Is Significant'] = (df[logfoldchange_cols[0]].abs() > 0.585) & (df[pval_cols[0]] < 10e-6)

df.to_excel(writer, sheet_name=sanitize_sheet_name(groups[0]))
        
writer.close()

# All other tissues

In [58]:
regen = adata[adata.obs['treg_celltypes'].isin(['0:Regenerative'])].copy()
regen.obs['treg_celltypes']

CGTTCTGAGATCGGGT-1@gvhd_spleen    0:Regenerative
CCGTTCACAGTCAGCC-1@gvhd_spleen    0:Regenerative
TCCCGATAGTGCGATG-1@ss_skin        0:Regenerative
CGTGTCTTCCACGCAG-1@gvhd_spleen    0:Regenerative
TGACGGCGTTGCGTTA-1@ss_skin        0:Regenerative
                                       ...      
ACTGTCCAGCTAGTCT-1@ss_bm          0:Regenerative
TTGCGTCCACGACGAA-1@hct_spleen     0:Regenerative
GATCGCGAGCGAGAAA-1@hct_bm         0:Regenerative
TTGGCAAAGGCCCTTG-1@ss_liver       0:Regenerative
AAGGAGCGTCTAGGTT-1@gvhd_spleen    0:Regenerative
Name: treg_celltypes, Length: 5354, dtype: category
Categories (1, object): ['0:Regenerative']

In [59]:
regen_nonss = regen[~regen.obs['condition'].isin(['ss'])].copy()
regen_nonss 

AnnData object with n_obs × n_vars = 3082 × 23867
    obs: 'condition', 'tissue', 'sample', 'cell_type', 'tissues_ordered', 'spectra50', 'treg_regen', 'treg_supress', 'treg_naive', 'treg_celltypes'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'cell_type_colors', 'condition_colors', 'log1p', 'neighbors', 'pca', 'sample_colors', 'tissue_colors', 'tissues_ordered_colors', 'treg_celltypes_colors', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'raw_data'
    obsp: 'connectivities', 'distances'

In [62]:
tissues = regen_nonss.obs["tissue"].unique().tolist()
tissues

['spleen', 'thymus', 'bm', 'liver', 'skin']

In [64]:
start_nfile = 5
nfile = start_nfile

for tissue in tissues:
    out_path = os.path.join(deg_outputs, f"{nfile}_{tissue}_treg_vs_rest.xlsx")
    with pd.ExcelWriter(out_path, engine="xlsxwriter") as writer:

        sc.tl.rank_genes_groups(
            regen_nonss,
            "tissue",
            groups=[f"{tissue}"],
            reference="rest",
            method="t-test",
            use_raw=False,
        )

        result = regen_nonss.uns["rank_genes_groups"]
        groups = result["names"].dtype.names

        df = pd.DataFrame({
            group + "_" + key[:1]: result[key][group]
            for group in groups
            for key in ["names", "scores", "logfoldchanges", "pvals_adj"]
        })

        logfoldchange_cols = [c for c in df.columns if c.endswith("_l")]
        pval_cols = [c for c in df.columns if c.endswith("_p")]

        if logfoldchange_cols and pval_cols:
            df["Is Significant"] = (df[logfoldchange_cols[0]].abs() > 0.585) & (df[pval_cols[0]] < 10e-6)

        df.to_excel(writer, sheet_name=sanitize_sheet_name(groups[0]), index=False)

    nfile += 1
